In [62]:
import cv2
import json
import requests
import base64
from pathlib import Path
from pydantic import BaseModel
from typing import List, Optional
from IPython.display import clear_output, display

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import matplotlib.image as mpimg
import os
import pickle

In [45]:
class ObjectText(BaseModel):
    Title: str
    Description: str

In [46]:
def get_llava_annotation(image_path,messages,detected_objects):
    img = mpimg.imread(image_path)
    image_bytes = open(image_path, "rb").read()
    
    messages.append({
        "role":"user",
        "content":f"""
        Create a description of the following <image>.
        List of detected objects: {detected_objects}
        Use the following structure:
        - Title
        - Description
        Title should be unique and specific for the provided image, distinguishing it from eventual similar images.
        Use the json format.
        """,
        "images":[base64.b64encode(image_bytes).decode("utf-8")
    ]})

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "gemma3",
            "messages": messages,
            "stream": False,
            "format": ObjectText.model_json_schema(),
            "options": {
            "num_predict": 1024,
            "seed": 4325
        }
        }
    )
    resp_json = response.json()
    messages.append(resp_json["message"])
    return (messages,resp_json["message"]["content"])

In [49]:
def extract_coco_domain(domain_name):
    with open(f"./downloaded_json/{domain_name}_gt.json","r") as f:
        coco_data = json.loads(f.read())
    coco_data_flat = {
        im["file_name"]:{
            "width":im["width"],
            "height":im["height"],
            "annotations":[],
            "categories":set()
        } for im in coco_data["images"]
    }
    image_id_to_file_name = {im["id"]:im["file_name"] for im in coco_data["images"]}
    cat_id_to_cat_name = {cat["id"]:cat["name"] for cat in coco_data["categories"]}
    for ann in coco_data["annotations"]:
        image_id = ann["image_id"]
        cat_id = ann["category_id"]
        fname = image_id_to_file_name[image_id]
        cname = cat_id_to_cat_name[cat_id]
        #coco_data_flat[fname]["annotations"].append({"cat":cname,"bbox":ann["bbox"]})
        coco_data_flat[fname]["categories"].add(cname)

    image_names = list(coco_data_flat.keys())
    return (coco_data_flat,image_names)

In [60]:
def gen_descriptions(domain_name,coco_data_flat):
    dir_path = f"./downloaded_images/{domain_name}/data"
    entries = os.listdir(dir_path)
    result = dict()
    cnt = 0
    total_cnt = len(entries)
    for entry in entries:
        file_name = os.path.join(dir_path, entry)
        if os.path.isfile(file_name):
            try:
                #clear_output(wait=True)
                #print(entry)
                messages = [{
                    "role":"system",
                    "content":"""
                    You are a helpful assistant.
                    """}]
                detections = ",".join(list(coco_data_flat[entry]["categories"]))
                (messages,annotation) = get_llava_annotation(file_name,messages,detections)
                result[entry] = annotation
                #print(annotation)
                cnt += 1
                if cnt % 100 == 0:
                    print(f"{cnt}/{total_cnt}")
                #if cnt == 1:
                #    break
            except:
                print("Error")
    return result

In [58]:
def gen_domain_html(domain_name,result):
    dir_path = f"./downloaded_images/{domain_name}/data"
    f = open(f"./descriptions/{domain_name}_gen.html","wt")
    f.write(f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>{domain_name}</title>
        </head>
        <body>
        <table>
        """)
    for entry in result.keys():
        file_name = os.path.join(dir_path, entry)
        img = cv2.imread(file_name)
        w,h,_ = img.shape
        img = cv2.resize(img, dsize=(150, round(150*w/h)))
        _, buffer = cv2.imencode('.jpg', img)
        img_base64 = base64.b64encode(buffer)
        img_base64_str = img_base64.decode('utf-8')
        desc = result[entry]
        f.write(f"""<tr><td colspan="2">
        {entry}
        </td></tr>
        <tr>
        <td>
        <img style='display:block;' id='{entry}'
           src='data:image/jpeg;base64, {img_base64_str}' />
        </td><td>{desc}</td></tr>""")

    f.write("""
    </table></body></html>
    """)
    f.close()

In [ ]:
domains = os.listdir("./downloaded_images")
processed_domains = os.listdir("./descriptions")
for domain in domains:
    if f"{domain}.pkl" in processed_domainss:
        print(f"Already processed {domain}")
        continue
    clear_output(wait=True)
    print(domain)
    (coco_data_flat,image_names) = extract_coco_domain(domain)
    result = gen_descriptions(domain,coco_data_flat)
    gen_domain_html(domain,result)
    with open(f"./descriptions/{domain}.pkl", "wb") as f:
        pickle.dump(result, f)

In [26]:

# Save the object to a file
with open('baseball.pkl', 'wb') as f:
    pickle.dump(result, f)

In [474]:
import pickle

with open('baseball.pkl', 'rb') as f:
    result = pickle.load(f)

In [10]:
result['pexels_5203106.jpeg']

[{'annotations': [{'object_id': 1,
    'category': 'person',
    'bounding_box': {'left': 0.245,
     'bottom': 0.375,
     'right': 0.649,
     'top': 0.678},
    'confidence': 0.981}]},
 {'annotations': [{'object_id': 1,
    'category': 'person',
    'bounding_box': {'left': 0.345,
     'bottom': 0.277,
     'right': 0.619,
     'top': 0.724},
    'confidence': 0.988}]},
 {'annotations': [{'object_id': 1,
    'category': 'person',
    'bounding_box': {'left': 0.245,
     'bottom': 0.375,
     'right': 0.649,
     'top': 0.678},
    'confidence': 0.981}]}]